In [1]:
import sqlite3
import pandas as pd

In [2]:
con = sqlite3.connect("vp_data2_isikud.db")
cur = con.cursor()

In [3]:
# transaktsioonide andmebaasi lisamine (v33)
cur.execute('ATTACH DATABASE "v33.db" AS v33')

In [4]:
# määruste andmebaasi lisamine (maarused)
cur.execute('ATTACH DATABASE "db_maarused.db" AS maarused')

## Create new transaction table

### juurde lisada veerud koht ja elus, kus on listide põhjal otsus sõna kohta

In [6]:
%%time

cur.execute("""
CREATE TABLE transaction_v2 as
SELECT * FROM 'transaction'
""")

CPU times: user 13.4 s, sys: 2.71 s, total: 16.1 s
Wall time: 17 s


In [7]:

cur.execute("""
ALTER TABLE transaction_v2
ADD koht VARCHAR(255) DEFAULT 'UNK'
""")

con.commit()

In [8]:
cur.execute("""
ALTER TABLE transaction_v2
ADD elus VARCHAR(255) DEFAULT 'UNK'
""")

con.commit()

In [9]:
%%time

cur.execute("""
UPDATE transaction_v2
SET elus = 'YES'
WHERE lower(transaction_v2.lemma) in (select lower(lemma) from elus_bio_v1)
""")

con.commit()

CPU times: user 21 s, sys: 6.51 s, total: 27.5 s
Wall time: 1min 55s


In [10]:
%%time

cur.execute("""
UPDATE transaction_v2
SET koht = 'YES'
WHERE lower(transaction_v2.lemma) in (select lower(lemma) from kohad_v1)
""")

con.commit()

CPU times: user 16.9 s, sys: 5.64 s, total: 22.6 s
Wall time: 1min 5s


In [9]:
con.close()

## kontroll

In [11]:
res2 = cur.execute("""
SELECT * from transaction_v2 
where koht='YES'
limit 20
""")

for i, e in enumerate(res2):
    print(e)

(215, 152, 19, -3, 'nsubj', 'uks', 'uks', 'com,nom,sg', None, 'S', 'YES', 'UNK')
(249, 172, 6, 2, 'obj', 'eriala', 'eriala', 'com,nom,sg', None, 'S', 'YES', 'UNK')
(428, 274, 7, -2, 'obl', 'kohvikus', 'kohvik', 'com,in,sg', None, 'S', 'YES', 'UNK')
(458, 295, 3, -1, 'obl', 'Lennujaamas', 'lennujaam', 'com,in,sg', None, 'S', 'YES', 'UNK')
(463, 296, 11, 1, 'obl', 'liiklusummikusse', 'liiklusummik', 'com,ill,sg', None, 'S', 'YES', 'UNK')
(478, 301, 3, 1, 'obl', 'lennujaama', 'lennujaam', 'com,gen,sg', None, 'S', 'YES', 'UNK')
(572, 355, 10, 2, 'obl', 'toas', 'tuba', 'com,in,sg', None, 'S', 'YES', 'UNK')
(580, 356, 3, 1, 'obl', 'tuppa', 'tuba', 'adit,com,sg', None, 'S', 'YES', 'UNK')
(693, 428, 15, 2, 'obl', 'rõdult', 'rõdu', 'abl,com,sg', None, 'S', 'YES', 'UNK')
(707, 438, 11, 1, 'obl', 'maja', 'maja', 'com,gen,sg', None, 'S', 'YES', 'UNK')
(773, 481, 30, -1, 'obl', 'linnal', 'linn', 'ad,com,sg', None, 'S', 'YES', 'UNK')
(779, 484, 4, 1, 'obj', 'maja', 'maja', 'com,part,sg', None, 'S', 

In [12]:
res2 = cur.execute("""
SELECT * from transaction_v2 
where elus='YES'
limit 20
""")

for i, e in enumerate(res2):
    print(e)

(4, 3, 1, -3, 'obj', 'Bändi', 'bänd', 'adit,com,sg', None, 'S', 'UNK', 'YES')
(9, 4, 4, -2, 'nsubj', 'solist', 'solist', 'com,nom,sg', None, 'S', 'UNK', 'YES')
(19, 10, 4, 1, 'obl', 'sul', 'sina', 'ad,sg', None, 'P', 'UNK', 'YES')
(21, 11, 1, -1, 'nsubj', 'Ma', 'mina', 'nom,sg', None, 'P', 'UNK', 'YES')
(23, 11, 6, 2, 'obl', 'juhul', 'juht', 'ad,com,sg', None, 'S', 'UNK', 'YES')
(24, 14, 8, 1, 'obj', 'bändi', 'bänd', 'com,part,sg', None, 'S', 'UNK', 'YES')
(33, 22, 7, -3, 'nsubj', 'ma', 'mina', 'nom,sg', None, 'P', 'UNK', 'YES')
(43, 27, 9, -2, 'nsubj', 'ma', 'mina', 'nom,sg', None, 'P', 'UNK', 'YES')
(50, 30, 1, -1, 'nsubj', 'Ma', 'mina', 'nom,sg', None, 'P', 'UNK', 'YES')
(56, 31, 3, 2, 'obl', 'diskorina', 'diskor', 'com,es,sg', None, 'S', 'UNK', 'YES')
(63, 35, 4, 1, 'nsubj', 'ansamblid', 'ansambel', 'com,nom,pl', None, 'S', 'UNK', 'YES')
(80, 49, 16, -1, 'nsubj', 'mina', 'mina', 'nom,sg', None, 'P', 'UNK', 'YES')
(82, 51, 5, -4, 'nsubj', 'me', 'mina', 'nom,pl', None, 'P', 'UNK', 'Y

In [13]:
# elus

res2 = cur.execute("""
SELECT count(*) from transaction_v2 
where elus='YES'
""")

for i, e in enumerate(res2):
    print(e)

(6044615,)


In [14]:
# koht

res2 = cur.execute("""
SELECT count(*) from transaction_v2 
where koht='YES'
""")

for i, e in enumerate(res2):
    print(e)

(1449124,)


In [15]:
# märkimata

res2 = cur.execute("""
SELECT count(*) from transaction_v2 
where koht='UNK' and elus='UNK'
""")

for i, e in enumerate(res2):
    print(e)

(46587774,)


In [5]:
# root on mõlemat

res2 = cur.execute("""
SELECT count(*) from transaction_v2 
where koht='YES' and elus='YES'
""")

for i, e in enumerate(res2):
    print(e)

(31014,)


In [7]:
query = """

SELECT *
from transaction_v2 
where koht='YES' and elus='YES' and lemma = 'kodutu'
"""

s = pd.read_sql_query(query, con)
s

,id,head_id,loc,loc_rel,deprel,form,lemma,feats,parent_loc,pos,koht,elus
0,3722,2159,17,2,obl,kodutute,kodutu,"gen,pl,pos",None,A,YES,YES
1,111566,61479,15,1,xcomp,kodutuks,kodutu,"com,sg,tr",None,S,YES,YES
2,134667,74239,24,-1,xcomp,kodutuks,kodutu,"com,sg,tr",None,S,YES,YES
3,135060,74427,10,1,ccomp,kodutu,kodutu,"nom,pos,sg",None,A,YES,YES
4,146338,80683,6,-1,nsubj,kodutud,kodutu,"com,nom,pl",None,S,YES,YES
...,...,...,...,...,...,...,...,...,...,...,...,...
1103,52880215,29311442,6,1,obl,kodutu,kodutu,"com,gen,sg",None,S,YES,YES
1104,52970591,29365530,4,-1,obl,kodutul,kodutu,"ad,com,sg",None,S,YES,YES
1105,53667546,29830683,4,-1,nsubj,kodutud,kodutu,"com,nom,pl",None,S,YES,YES
1106,53774893,29897888,5,1,obj,kodutu,kodutu,"com,nom,sg",None,S,YES,YES


In [8]:
query = """

SELECT *
from transaction_v2 
where koht==elus and deprel = 'obl'
"""

s = pd.read_sql_query(query, con)
s

,id,head_id,loc,loc_rel,deprel,form,lemma,feats,parent_loc,pos,koht,elus
0,1,2,3,-1,obl,lõpus,lõpp,"com,in,sg",None,S,UNK,UNK
1,3,2,6,2,obl,1.,1.,"<?>,ord,roman",None,N,UNK,UNK
2,7,3,12,1,obl,keeltele,keel,"all,com,pl",None,S,UNK,UNK
3,15,6,4,1,obl,purukspeksmisega,purukspeksmine,"com,kom,sg",None,S,UNK,UNK
4,22,11,3,1,obl,tundidest,tund,"com,el,pl",None,S,UNK,UNK
...,...,...,...,...,...,...,...,...,...,...,...,...
10534920,54050467,30078975,9,1,obl,perse,perse,"adit,com,sg",None,S,UNK,UNK
10534921,54050473,30078981,33,1,obl,tegelt,tege,"abl,com,sg",None,S,UNK,UNK
10534922,54050477,30078982,8,2,obl,temaga,tema,"kom,sg",None,P,UNK,UNK
10534923,54050478,30078982,9,3,obl,tylli,tyll,"adit,com,sg",None,S,UNK,UNK
